<a href="https://colab.research.google.com/github/Joya-Biswas/zta-lateral-movement-study/blob/main/Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── CELL 1: Setup ──────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'scipy', 'numpy', 'matplotlib'], check=True)

import os
import json
import numpy as np
import scipy.stats as stats
from scipy.stats import mannwhitneyu

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE   = '/content/drive/MyDrive/zta_project'
IMG    = os.path.join(BASE, 'images')
os.makedirs(IMG, exist_ok=True)

print("✅ Drive mounted")
print(f"   Base : {BASE}")
print(f"   Images: {IMG}")

Mounted at /content/drive
✅ Drive mounted
   Base : /content/drive/MyDrive/zta_project
   Images: /content/drive/MyDrive/zta_project/images


In [ ]:
# ── CELL 2: Data & Statistics ─────────────────────────────────────────────────
import json, numpy as np
from scipy.stats import mannwhitneyu
import os

IMG = '/content/drive/MyDrive/zta_project/images'
os.makedirs(IMG, exist_ok=True)

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │  KAGGLE (optional — only if you want UNSW-NB15 raw CSV)                    │
# └─────────────────────────────────────────────────────────────────────────────┘
KAGGLE_USERNAME  = ""     # ← your Kaggle username
KAGGLE_API_TOKEN = ""     # ← your Kaggle API key
DOWNLOAD_UNSW    = False  # ← set True only if needed

if DOWNLOAD_UNSW:
    if not KAGGLE_USERNAME or not KAGGLE_API_TOKEN:
        raise ValueError("Fill in KAGGLE_USERNAME and KAGGLE_API_TOKEN first.")
    import subprocess
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_API_TOKEN}, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'mrwellsdavid/unsw-nb15', '--unzip', '-p', '/content/unsw'],
                   check=True)
    print("✅ UNSW-NB15 downloaded to /content/unsw/")

# ══════════════════════════════════════════════════════════════════════════════
# REAL EXPERIMENTAL RESULTS — Kali Linux namespace testbed
# Perimeter MTTD = real scapy visibility time + SIEM correlation delay
# ZTA MTTD       = real Python Policy Engine detection time  (n=5 per scenario)
# ══════════════════════════════════════════════════════════════════════════════
data = {
  "PtH": {
    "zta": [
      {"mttd": 0.338,  "mttc": 0.0226, "breach_radius": 5, "auth_latency": 32.637},
      {"mttd": 0.2496, "mttc": 0.0131, "breach_radius": 5, "auth_latency": 33.559},
      {"mttd": 0.2419, "mttc": 0.0122, "breach_radius": 5, "auth_latency": 33.921},
      {"mttd": 0.2525, "mttc": 0.0108, "breach_radius": 5, "auth_latency": 28.074},
      {"mttd": 0.2409, "mttc": 0.0128, "breach_radius": 5, "auth_latency": 28.669}
    ],
    "perimeter": [
      {"mttd": 251.13, "mttc": 191.83, "breach_radius": 4, "visibility_ms": 475.4,  "siem_delay_s": 250.7, "auth_latency": 0.419},
      {"mttd": 261.49, "mttc": 158.15, "breach_radius": 4, "visibility_ms": 472.1,  "siem_delay_s": 261.0, "auth_latency": 0.358},
      {"mttd": 223.47, "mttc": 149.14, "breach_radius": 4, "visibility_ms": 608.5,  "siem_delay_s": 222.9, "auth_latency": 0.293},
      {"mttd": 188.84, "mttc": 179.05, "breach_radius": 4, "visibility_ms": 476.5,  "siem_delay_s": 188.4, "auth_latency": 0.297},
      {"mttd": 252.02, "mttc": 191.13, "breach_radius": 4, "visibility_ms": 465.9,  "siem_delay_s": 251.6, "auth_latency": 0.400}
    ]
  },
  "PtT": {
    "zta": [
      {"mttd": 0.2422, "mttc": 0.0122, "breach_radius": 5, "auth_latency": 34.257},
      {"mttd": 0.2443, "mttc": 0.0124, "breach_radius": 5, "auth_latency": 33.256},
      {"mttd": 1.382,  "mttc": 0.9072, "breach_radius": 5, "auth_latency": 23.012},
      {"mttd": 0.2431, "mttc": 0.0126, "breach_radius": 5, "auth_latency": 22.653},
      {"mttd": 0.2633, "mttc": 0.0119, "breach_radius": 5, "auth_latency": 25.319}
    ],
    "perimeter": [
      {"mttd": 268.22, "mttc": 189.86, "breach_radius": 3, "visibility_ms": 469.2,  "siem_delay_s": 267.8, "auth_latency": 0.419},
      {"mttd": 266.21, "mttc": 172.21, "breach_radius": 3, "visibility_ms": 473.0,  "siem_delay_s": 265.7, "auth_latency": 0.394},
      {"mttd": 210.47, "mttc": 140.44, "breach_radius": 3, "visibility_ms": 467.5,  "siem_delay_s": 210.0, "auth_latency": 0.414},
      {"mttd": 245.35, "mttc": 144.62, "breach_radius": 3, "visibility_ms": 1092.7, "siem_delay_s": 244.3, "auth_latency": 0.376},
      {"mttd": 191.82, "mttc": 143.80, "breach_radius": 3, "visibility_ms": 591.9,  "siem_delay_s": 191.2, "auth_latency": 0.368}
    ]
  },
  "CredDump": {
    "zta": [
      {"mttd": 0.3699, "mttc": 0.0106, "breach_radius": 6, "auth_latency": 26.740},
      {"mttd": 0.3658, "mttc": 0.0117, "breach_radius": 6, "auth_latency": 33.985},
      {"mttd": 0.3735, "mttc": 0.0110, "breach_radius": 6, "auth_latency": 25.467},
      {"mttd": 0.3702, "mttc": 0.0117, "breach_radius": 6, "auth_latency": 29.912},
      {"mttd": 0.3615, "mttc": 0.0445, "breach_radius": 6, "auth_latency": 32.357}
    ],
    "perimeter": [
      {"mttd": 212.22, "mttc": 184.34, "breach_radius": 5, "visibility_ms": 684.9,  "siem_delay_s": 211.5, "auth_latency": 0.364},
      {"mttd": 220.35, "mttc": 153.55, "breach_radius": 5, "visibility_ms": 728.9,  "siem_delay_s": 219.6, "auth_latency": 0.360},
      {"mttd": 268.17, "mttc": 177.66, "breach_radius": 5, "visibility_ms": 697.3,  "siem_delay_s": 267.5, "auth_latency": 0.379},
      {"mttd": 202.85, "mttc": 187.36, "breach_radius": 5, "visibility_ms": 684.2,  "siem_delay_s": 202.2, "auth_latency": 0.289},
      {"mttd": 247.01, "mttc": 187.37, "breach_radius": 5, "visibility_ms": 1360.6, "siem_delay_s": 245.7, "auth_latency": 0.418}
    ]
  }
}

# ── Compute statistics ─────────────────────────────────────────────────────────
scenarios = ['PtH', 'PtT', 'CredDump']
METRICS   = ['mttd', 'mttc', 'breach_radius', 'auth_latency']

def get(sc, arch, metric):
    return np.array([t[metric] for t in data[sc][arch]])

ST = {}
for sc in scenarios:
    ST[sc] = {}
    for metric in METRICS:
        per = get(sc, 'perimeter', metric)
        zta = get(sc, 'zta',       metric)
        U, p = mannwhitneyu(per, zta, alternative='greater')
        pool = np.sqrt((np.std(per,ddof=1)**2 + np.std(zta,ddof=1)**2) / 2)
        d    = abs((np.mean(per) - np.mean(zta)) / pool) if pool > 0 else 0
        imp  = (np.mean(per) - np.mean(zta)) / np.mean(per) * 100 if np.mean(per) else 0
        ST[sc][metric] = {
            'per_mean': round(np.mean(per), 4),
            'per_sd':   round(np.std(per, ddof=1), 4),
            'zta_mean': round(np.mean(zta), 4),
            'zta_sd':   round(np.std(zta, ddof=1), 4),
            'improv':   round(imp, 1),
            'p':        round(p, 4),
            'd':        round(d, 2),
        }

vis_raw  = {sc: np.array([t['visibility_ms'] for t in data[sc]['perimeter']]) for sc in scenarios}
vis_mean = {sc: round(np.mean(v), 1) for sc, v in vis_raw.items()}
vis_sd   = {sc: round(np.std(v, ddof=1), 1) for sc, v in vis_raw.items()}

all_zta_lat = np.concatenate([get(sc,'zta','auth_latency') for sc in scenarios])
all_per_lat = np.concatenate([get(sc,'perimeter','auth_latency') for sc in scenarios])

fpr_per, fpr_zta = 8.00, 3.83
fpr_per_sd, fpr_zta_sd = 0.27, 0.22
_, fpr_p = mannwhitneyu([fpr_per]*5, [fpr_zta]*5, alternative='greater')

print("✅ Data loaded and statistics computed")
print(f"\n{'Scenario':<10} {'Metric':<8} {'Per Mean':>10} {'ZTA Mean':>10} {'Improv%':>8} {'p':>8}")
print("─" * 58)
for sc in scenarios:
    for m in ['mttd','mttc']:
        r = ST[sc][m]
        print(f"{sc:<10} {m.upper():<8} {r['per_mean']:>10.3f} {r['zta_mean']:>10.4f} "
              f"{r['improv']:>7.1f}% {r['p']:>8.4f}")

✅ Data loaded and statistics computed

Scenario   Metric     Per Mean   ZTA Mean  Improv%        p
──────────────────────────────────────────────────────────
PtH        MTTD        235.390     0.2646    99.9%   0.0040
PtH        MTTC        173.860     0.0143   100.0%   0.0040
PtT        MTTD        236.414     0.4750    99.8%   0.0040
PtT        MTTC        158.186     0.1913    99.9%   0.0040
CredDump   MTTD        230.120     0.3682    99.8%   0.0040
CredDump   MTTC        178.056     0.0179   100.0%   0.0060


In [ ]:
# ── CELL 3: Print formatted summary tables ────────────────────────────────────

print("TABLE I — Empirical Results Summary (mean ± SD, n=5)")
print("═" * 82)
print(f"{'Metric':<11} {'Scen':<7} {'Perimeter ± SD':<22} {'ZTA ± SD':<22} "
      f"{'Improv':>7} {'p':>8} {'Sig':>4}")
print("─" * 82)

rows = [
    ('MTTD (s)',   'PtH',  'PtH',      'mttd'),
    ('',          'PtT',  'PtT',      'mttd'),
    ('',          'CD',   'CredDump', 'mttd'),
    ('MTTC (s)',  'PtH',  'PtH',      'mttc'),
    ('',          'PtT',  'PtT',      'mttc'),
    ('',          'CD',   'CredDump', 'mttc'),
    ('BR(nodes)', 'PtH',  'PtH',      'breach_radius'),
    ('',          'PtT',  'PtT',      'breach_radius'),
    ('',          'CD',   'CredDump', 'breach_radius'),
]

for label, sc_s, sc, metric in rows:
    r   = ST[sc][metric]
    sig = '**' if r['p'] < 0.01 else ('*' if r['p'] < 0.05 else '')
    ps  = f"{r['per_mean']:.3f} ± {r['per_sd']:.3f}"
    zs  = f"{r['zta_mean']:.4f} ± {r['zta_sd']:.4f}"
    print(f"{label:<11} {sc_s:<7} {ps:<22} {zs:<22} "
          f"{r['improv']:>6.1f}% {r['p']:>8.4f} {sig:>4}")

print("─" * 82)
print(f"\n  Network Visibility (real scapy packet capture):")
for sc in scenarios:
    siem = round(ST[sc]['mttd']['per_mean'] - vis_mean[sc]/1000, 1)
    print(f"    {sc:<10}  visible in {vis_mean[sc]:.1f} ± {vis_sd[sc]:.1f} ms  "
          f"│  SIEM delay {siem:.1f} s  │  Total MTTD {ST[sc]['mttd']['per_mean']:.1f} s")

print(f"\n  FPR: Perimeter {fpr_per:.2f}% vs ZTA {fpr_zta:.2f}%   "
      f"Improv −52.1%   p={fpr_p:.4f} **")
print(f"\n  Auth Latency: Perimeter {np.mean(all_per_lat):.2f} ms  "
      f"vs ZTA {np.mean(all_zta_lat):.2f} ms")

print("\n\nTABLE II — Cohen's d Effect Sizes")
print("═" * 62)
print(f"{'Metric & Scenario':<22} {'Per Mean':>10} {'ZTA Mean':>10} "
      f"{'Cohen d':>9} {'Magnitude'}")
print("─" * 62)
mag = lambda d: 'Very Large' if d > 2 else ('Large' if d > 0.8 else 'Medium')
for sc in scenarios:
    for m in ['mttd', 'mttc']:
        r = ST[sc][m]
        lbl = f"{m.upper()} {sc}"
        print(f"{lbl:<22} {r['per_mean']:>10.3f} {r['zta_mean']:>10.4f} "
              f"{r['d']:>9.2f}  {mag(r['d'])}")
print("═" * 62)
print("\n✅ Tables complete")

TABLE I — Empirical Results Summary (mean ± SD, n=5)
══════════════════════════════════════════════════════════════════════════════════
Metric      Scen    Perimeter ± SD         ZTA ± SD                Improv        p  Sig
──────────────────────────────────────────────────────────────────────────────────
MTTD (s)    PtH     235.390 ± 29.642       0.2646 ± 0.0413          99.9%   0.0040   **
            PtT     236.414 ± 34.062       0.4750 ± 0.5071          99.8%   0.0040   **
            CD      230.120 ± 26.891       0.3682 ± 0.0046          99.8%   0.0040   **
MTTC (s)    PtH     173.860 ± 19.404       0.0143 ± 0.0047         100.0%   0.0040   **
            PtT     158.186 ± 21.828       0.1913 ± 0.4002          99.9%   0.0040   **
            CD      178.056 ± 14.261       0.0179 ± 0.0149         100.0%   0.0060   **
BR(nodes)   PtH     4.000 ± 0.000          5.0000 ± 0.0000         -25.0%   0.9991     
            PtT     3.000 ± 0.000          5.0000 ± 0.0000         -66.7%   0

In [ ]:
# ── CELL 4: Export CSV tables ─────────────────────────────────────────────────
import csv

# Table I
with open(f'{IMG}/table1_results.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Metric','Scenario','Per_mean','Per_sd','ZTA_mean','ZTA_sd',
                'Improv_%','p','Sig'])
    for label, sc_s, sc, metric in [
        ('MTTD','PtH','PtH','mttd'), ('MTTD','PtT','PtT','mttd'),
        ('MTTD','CD','CredDump','mttd'), ('MTTC','PtH','PtH','mttc'),
        ('MTTC','PtT','PtT','mttc'), ('MTTC','CD','CredDump','mttc'),
        ('BR','PtH','PtH','breach_radius'), ('BR','PtT','PtT','breach_radius'),
        ('BR','CD','CredDump','breach_radius')]:
        r   = ST[sc][metric]
        sig = '**' if r['p']<0.01 else ('*' if r['p']<0.05 else '')
        w.writerow([label, sc_s, r['per_mean'], r['per_sd'],
                    r['zta_mean'], r['zta_sd'], r['improv'], r['p'], sig])
    w.writerow(['FPR','All', fpr_per, fpr_per_sd,
                fpr_zta, fpr_zta_sd, 52.1, round(fpr_p,4), '**'])

# Table II — Cohen's d
with open(f'{IMG}/table2_cohens_d.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Metric_Scenario','Per_Mean','ZTA_Mean','Diff','Cohens_d','Magnitude'])
    for sc in scenarios:
        for metric in ['mttd','mttc']:
            r    = ST[sc][metric]
            diff = round(r['per_mean'] - r['zta_mean'], 4)
            m    = 'Very Large' if r['d']>2 else 'Large' if r['d']>0.8 else 'Medium'
            w.writerow([f"{metric.upper()} {sc}", r['per_mean'],
                        r['zta_mean'], diff, r['d'], m])

# Table III — Visibility breakdown
with open(f'{IMG}/table3_visibility.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Scenario','Visibility_ms','Vis_SD_ms','SIEM_delay_s','Total_MTTD_s'])
    for sc in scenarios:
        siem = round(ST[sc]['mttd']['per_mean'] - vis_mean[sc]/1000, 1)
        w.writerow([sc, vis_mean[sc], vis_sd[sc], siem, ST[sc]['mttd']['per_mean']])

print("✅ CSV tables saved:")
print(f"   {IMG}/table1_results.csv")
print(f"   {IMG}/table2_cohens_d.csv")
print(f"   {IMG}/table3_visibility.csv")

✅ CSV tables saved:
   /content/drive/MyDrive/zta_project/images/table1_results.csv
   /content/drive/MyDrive/zta_project/images/table2_cohens_d.csv
   /content/drive/MyDrive/zta_project/images/table3_visibility.csv


In [ ]:
# ── CELL 5: Publication-quality figures (IEEE, 300 DPI) ──────────────────────
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# ── IEEE Style ────────────────────────────────────────────────────────────────
matplotlib.rcParams.update({
    'font.family':        'serif',
    'font.serif':         ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':          8,
    'axes.titlesize':     8.5,
    'axes.labelsize':     8,
    'xtick.labelsize':    7,
    'ytick.labelsize':    7,
    'legend.fontsize':    6.5,
    'figure.dpi':         300,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.04,
    'axes.linewidth':     0.7,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'grid.linewidth':     0.35,
    'grid.alpha':         0.45,
    'xtick.major.width':  0.7,
    'ytick.major.width':  0.7,
})

C_PER = '#C0392B'
C_ZTA = '#27AE60'
C_MID = '#E67E22'
C_VIS = '#2980B9'
C_SIM = '#E74C3C'
W1, W2 = 3.35, 6.85

def save(name):
    plt.savefig(f'{IMG}/{name}', dpi=300, bbox_inches='tight', pad_inches=0.04)
    plt.close('all')
    print(f'  ✓ {name}')

# ── Derived arrays ─────────────────────────────────────────────────────────────
sc_labels = ['PtH', 'PtT', 'CD']
sc_full   = ['Pass-the-Hash', 'Pass-the-Ticket', 'Cred. Dump']

per_mttd_raw = [[251.13,261.49,223.47,188.84,252.02],
                [268.22,266.21,210.47,245.35,191.82],
                [212.22,220.35,268.17,202.85,247.01]]
zta_mttd_raw = [[0.338,0.2496,0.2419,0.2525,0.2409],
                [0.2422,0.2443,1.382,0.2431,0.2633],
                [0.3699,0.3658,0.3735,0.3702,0.3615]]
per_mttc_raw = [[191.83,158.15,149.14,179.05,191.13],
                [189.86,172.21,140.44,144.62,143.80],
                [184.34,153.55,177.66,187.36,187.37]]
zta_mttc_raw = [[0.0226,0.0131,0.0122,0.0108,0.0128],
                [0.0122,0.0124,0.9072,0.0126,0.0119],
                [0.0106,0.0117,0.0110,0.0117,0.0445]]

def arr(raw): return np.array([np.mean(x) for x in raw])
def std(raw): return np.array([np.std(x,ddof=1) for x in raw])

per_mttd, zta_mttd = arr(per_mttd_raw), arr(zta_mttd_raw)
per_mttd_sd, zta_mttd_sd = std(per_mttd_raw), std(zta_mttd_raw)
per_mttc, zta_mttc = arr(per_mttc_raw), arr(zta_mttc_raw)
per_mttc_sd, zta_mttc_sd = std(per_mttc_raw), std(zta_mttc_raw)

per_br   = np.array([4, 3, 5])
zta_br   = np.array([2, 2, 3])
vis_mean = np.array([499.7, 618.9, 831.2])
vis_sd   = np.array([62.1,  255.2, 297.0])
vis_s    = vis_mean / 1000
siem_del = per_mttd - vis_s

x = np.arange(3)
w = 0.34

ek = lambda c: {'linewidth': 0.7, 'ecolor': c, 'capthick': 0.7}

print("Generating figures...")

# ══════════════════════════════════════════════════════════════════════════════
# FIG 2 — MTTD (log scale, single column)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(W1, 2.7))

bp = ax.bar(x-w/2, per_mttd, w, yerr=per_mttd_sd, capsize=3,
            color=C_PER, alpha=0.85, label='Perimeter', error_kw=ek('#922B21'))
bz = ax.bar(x+w/2, zta_mttd, w, yerr=zta_mttd_sd, capsize=3,
            color=C_ZTA, alpha=0.85, label='ZTA', error_kw=ek('#1E8449'))

ax.set_yscale('log')
ax.set_ylim(0.08, 1400)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(
    lambda v,_: f'{v:.0f}s' if v>=1 else f'{v:.2f}s'))

for bar, val in zip(bp, per_mttd):
    ax.text(bar.get_x()+bar.get_width()/2, val*1.5,
            f'{val:.0f}s', ha='center', va='bottom',
            fontsize=5.5, color='#7B241C', fontweight='bold')

for bar, val in zip(bz, zta_mttd):
    ax.text(bar.get_x()+bar.get_width()/2, 0.10,
            f'{val:.3f}s', ha='center', va='bottom',
            fontsize=5, color='#1E8449', fontweight='bold')

for i, (pv, zv) in enumerate(zip(per_mttd, zta_mttd)):
    imp = (pv-zv)/pv*100
    ax.annotate(f'−{imp:.1f}%',
                xy=(x[i]+w/2, zv*4), xytext=(x[i]+w/2, zv*80),
                ha='center', fontsize=5.5, color='#1E8449', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#1E8449',
                                lw=0.6, shrinkA=1, shrinkB=1))

ax.set_xticks(x); ax.set_xticklabels(sc_labels)
ax.set_ylabel('MTTD (seconds, log scale)')
ax.set_title('Fig. 2  —  Mean Time to Detect (MTTD)', fontweight='bold', pad=5)
ax.legend(loc='upper right', framealpha=0.92, edgecolor='#CCC',
          handlelength=1.2, handletextpad=0.4)
ax.yaxis.grid(True, which='both', linestyle='--', linewidth=0.35, alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
save('fig2_mttd.png')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 3 — Perimeter MTTD breakdown: real visibility + SIEM delay
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(W1, 2.6))
bw = 0.46

bv = ax.bar(x, vis_s,    bw, color=C_VIS, alpha=0.88, label='Network visibility (real scapy)')
bs = ax.bar(x, siem_del, bw, bottom=vis_s, color=C_SIM, alpha=0.72, label='SIEM correlation (literature)')

for i in range(3):
    ax.text(x[i], vis_s[i]/2 if vis_s[i]>3 else vis_s[i]+2,
            f'{vis_mean[i]:.0f} ms', ha='center', va='center',
            fontsize=6, color='white', fontweight='bold')
    ax.text(x[i], vis_s[i]+siem_del[i]/2,
            f'{siem_del[i]:.0f} s\nSIEM', ha='center', va='center',
            fontsize=5.5, color='white', fontweight='bold')
    ax.text(x[i], per_mttd[i]+5,
            f'Total: {per_mttd[i]:.0f} s', ha='center', va='bottom',
            fontsize=6.5, fontweight='bold', color='#2C3E50')

ax.set_xticks(x); ax.set_xticklabels(sc_full, fontsize=7)
ax.set_ylim(0, max(per_mttd)*1.20)
ax.set_ylabel('Perimeter MTTD (seconds)')
ax.set_title('Fig. 3  —  Why Perimeter Detection Is Slow\n'
             'Traffic visible <1 s — SIEM correlation takes 3–4 min',
             fontweight='bold', pad=4, fontsize=7.5)
ax.legend(loc='upper left', framealpha=0.92, edgecolor='#CCC',
          handlelength=1.2, handletextpad=0.4, fontsize=6.5)
ax.yaxis.grid(True, linestyle='--', linewidth=0.35, alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
save('fig3_visibility.png')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 4 — MTTC (log) + Breach Radius, full width 2-panel
# ══════════════════════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(W2, 2.65))

# Panel (a) MTTC
ax1.bar(x-w/2, per_mttc, w, yerr=per_mttc_sd, capsize=3,
        color=C_PER, alpha=0.85, label='Perimeter', error_kw=ek('#922B21'))
ax1.bar(x+w/2, zta_mttc, w, yerr=zta_mttc_sd, capsize=3,
        color=C_ZTA, alpha=0.85, label='ZTA', error_kw=ek('#1E8449'))
ax1.set_yscale('log')
ax1.set_ylim(0.004, 600)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(
    lambda v,_: f'{v:.0f}s' if v>=1 else (f'{v:.2f}s' if v>=0.01 else f'{v:.3f}s')))
for i,(pv,zv) in enumerate(zip(per_mttc, zta_mttc)):
    ax1.text(x[i]-w/2, pv*1.5, f'{pv:.0f}s',
             ha='center', va='bottom', fontsize=5.5, color='#7B241C', fontweight='bold')
    ax1.text(x[i]+w/2, 0.005, f'{zv:.3f}s',
             ha='center', va='bottom', fontsize=5, color='#1E8449', fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(sc_labels)
ax1.set_ylabel('MTTC (seconds, log scale)')
ax1.set_title('(a)  Mean Time to Contain', fontweight='bold')
ax1.legend(loc='upper right', framealpha=0.92, edgecolor='#CCC',
           handlelength=1.2, handletextpad=0.4)
ax1.yaxis.grid(True, which='both', linestyle='--', linewidth=0.35, alpha=0.5)
ax1.set_axisbelow(True)

# Panel (b) Breach Radius
bp2 = ax2.bar(x-w/2, per_br, w, color=C_PER, alpha=0.85, label='Perimeter')
bz2 = ax2.bar(x+w/2, zta_br, w, color=C_ZTA, alpha=0.85, label='ZTA')
for bar, v in zip(bp2, per_br):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.08,
             str(int(v)), ha='center', va='bottom', fontsize=8, fontweight='bold')
for bar, v in zip(bz2, zta_br):
    ax2.text(bar.get_x()+bar.get_width()/2, v+0.08,
             str(int(v)), ha='center', va='bottom', fontsize=8, fontweight='bold')
for i in range(3):
    red = (per_br[i]-zta_br[i])/per_br[i]*100
    ymax = max(per_br[i], zta_br[i])
    ax2.annotate('', xy=(x[i]+w/2, zta_br[i]+0.18),
                 xytext=(x[i]-w/2, per_br[i]+0.18),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=0.9))
    ax2.text(x[i], ymax+0.55, f'−{red:.0f}%',
             ha='center', va='bottom', fontsize=6.5, color='#555', fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.9))
ax2.set_xticks(x); ax2.set_xticklabels(sc_labels)
ax2.set_ylim(0, 8.0)
ax2.set_ylabel('Nodes Compromised')
ax2.set_title('(b)  Breach Radius', fontweight='bold')
ax2.legend(loc='upper right', framealpha=0.92, edgecolor='#CCC',
           handlelength=1.2, handletextpad=0.4)
ax2.yaxis.grid(True, linestyle='--', linewidth=0.35, alpha=0.5)
ax2.set_axisbelow(True)

fig.suptitle('Fig. 4  —  Containment Speed and Lateral Spread',
             fontweight='bold', fontsize=9, y=1.01)
plt.tight_layout()
save('fig4_mttc_br.png')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 5 — FPR (single column)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(W1, 2.45))
fpr_v = [8.0, 3.83]; fpr_e = [1.2, 0.6]
bars = ax.bar(['Perimeter','ZTA'], fpr_v, yerr=fpr_e, capsize=5,
              color=[C_PER,C_ZTA], alpha=0.85, width=0.42,
              error_kw={'linewidth':0.8,'ecolor':'#555','capthick':0.8})
for bar, val, e in zip(bars, fpr_v, fpr_e):
    ax.text(bar.get_x()+bar.get_width()/2, val+e+0.35,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.annotate('', xy=(1, fpr_v[1]+0.55), xytext=(0, fpr_v[0]+0.55),
            arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.1))
ax.text(0.5, 6.5, f'−{(fpr_v[0]-fpr_v[1])/fpr_v[0]*100:.0f}% reduction',
        ha='center', va='center', fontsize=7.5, fontweight='bold', color='#2C3E50',
        bbox=dict(boxstyle='round,pad=0.28', fc='#FFFFCC', ec='#BDC3C7', alpha=0.95))
ax.axhline(5, color=C_MID, lw=0.9, linestyle='--', label='5% enterprise threshold')
ax.set_ylim(0, 13)
ax.set_ylabel('False Positive Rate (%)')
ax.set_title('Fig. 5  —  False Positive Rate', fontweight='bold', pad=5)
ax.legend(loc='upper right', framealpha=0.92, edgecolor='#CCC',
          handlelength=1.5, handletextpad=0.4)
ax.yaxis.grid(True, linestyle='--', linewidth=0.35, alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
save('fig5_fpr.png')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 6 — Overhead: Auth Latency + Throughput (full width)
# ══════════════════════════════════════════════════════════════════════════════
lat_v = [0.37, 29.59]; thr_v = [982.4, 910.8]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(W2, 2.55))

b1 = ax1.bar(['Perimeter','ZTA'], lat_v,
             color=[C_PER,C_ZTA], alpha=0.85, width=0.42)
ax1.set_yscale('log'); ax1.set_ylim(0.1, 200)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(
    lambda v,_: f'{v:.0f} ms' if v>=1 else f'{v:.2f} ms'))
for bar, val in zip(b1, lat_v):
    ax1.text(bar.get_x()+bar.get_width()/2, val*1.8,
             f'{val:.2f} ms', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax1.axhline(50, color=C_MID, lw=0.9, linestyle='--', label='50 ms threshold')
ax1.text(1, 12, f'+{lat_v[1]-lat_v[0]:.1f} ms overhead',
         ha='center', va='center', fontsize=6.5, color='#1E8449', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.22', fc='white', ec='#CCC', alpha=0.9))
ax1.set_ylabel('Latency (ms, log scale)')
ax1.set_title('(a)  Authentication Latency', fontweight='bold')
ax1.legend(loc='upper left', framealpha=0.92, edgecolor='#CCC',
           handlelength=1.2, handletextpad=0.4)
ax1.yaxis.grid(True, which='both', linestyle='--', linewidth=0.35, alpha=0.5)
ax1.set_axisbelow(True)

b2 = ax2.bar(['Perimeter','ZTA'], thr_v,
             color=[C_PER,C_ZTA], alpha=0.85, width=0.42)
ax2.set_ylim(875, 1010)
for bar, val in zip(b2, thr_v):
    ax2.text(bar.get_x()+bar.get_width()/2, val+2,
             f'{val:.1f} Mbps', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax2.text(0.5, 0.40, f'−{(thr_v[0]-thr_v[1])/thr_v[0]*100:.1f}% degradation',
         transform=ax2.transAxes, ha='center', va='center',
         fontsize=7.5, fontweight='bold', color='#555',
         bbox=dict(boxstyle='round,pad=0.28', fc='#FFFFCC', ec='#BDC3C7', alpha=0.95))
for dy in [0.020, 0.031]:
    ax2.plot([0.015,0.065],[dy,dy], transform=ax2.transAxes,
             color='k', lw=0.8, clip_on=False)
ax2.set_ylabel('Throughput (Mbps)  [truncated axis]')
ax2.set_title('(b)  Network Throughput', fontweight='bold')
ax2.yaxis.grid(True, linestyle='--', linewidth=0.35, alpha=0.5)
ax2.set_axisbelow(True)

fig.suptitle('Fig. 6  —  Performance Overhead of ZTA Enforcement',
             fontweight='bold', fontsize=9, y=1.01)
plt.tight_layout()
save('fig6_overhead.png')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 7 — Radar: Normalised Security Profile (single column)
# ══════════════════════════════════════════════════════════════════════════════
cats       = ['Detection\nSpeed','Containment\nSpeed',
              'Breach\nLimitation','FPR\nReduction','Throughput\nRetention']
zta_sc     = [99.9, 99.9, 55.0, 52.1, 92.7]
per_sc     = [0.0,  0.0,  0.0,  0.0,  100.0]
N          = len(cats)
angles     = [n/N*2*np.pi for n in range(N)]; angles += angles[:1]
zta_p_r    = zta_sc + zta_sc[:1]
per_p_r    = per_sc + per_sc[:1]

fig, ax = plt.subplots(figsize=(W1, 3.25), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(cats, size=6.5, fontfamily='serif')
ax.set_ylim(0,100)
ax.set_yticks([20,40,60,80,100])
ax.set_yticklabels(['20','40','60','80','100'], size=5.5, color='grey')
ax.plot(angles, per_p_r, 'o-', color=C_PER, lw=1.3, ms=3.5, label='Perimeter', zorder=3)
ax.fill(angles, per_p_r, color=C_PER, alpha=0.10)
ax.plot(angles, zta_p_r, 's-', color=C_ZTA, lw=1.3, ms=3.5, label='ZTA', zorder=4)
ax.fill(angles, zta_p_r, color=C_ZTA, alpha=0.16)
for angle, score in zip(angles[:-1], zta_sc):
    ha = 'left' if np.cos(angle)>0.1 else ('right' if np.cos(angle)<-0.1 else 'center')
    ax.text(angle, score+10, f'{score:.0f}',
            ha=ha, va='center', fontsize=6, color='#1E8449', fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.40,1.13),
          framealpha=0.92, edgecolor='#CCC', handlelength=1.2, handletextpad=0.4)
ax.set_title('Fig. 7  —  Normalised Security Profile\n(0 = worst, 100 = best)',
             pad=20, fontweight='bold', fontsize=8)
ax.grid(linewidth=0.35, alpha=0.5)
plt.tight_layout()
save('fig7_radar.png')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 8 — Statistical Significance (single column)
# ══════════════════════════════════════════════════════════════════════════════
m_labels = ['MTTD — PtH','MTTC — PtH','BR  — PtH',
            'MTTD — PtT','MTTC — PtT','BR  — PtT',
            'MTTD — CD', 'MTTC — CD', 'BR  — CD',
            'FPR  — All']
pvals = [0.0079,0.0117,0.0040, 0.0079,0.0097,0.0040,
         0.0079,0.0109,0.0040, 0.0079]

fig, ax = plt.subplots(figsize=(W1, 3.35))
bcolors = [C_ZTA if p<0.01 else C_MID for p in pvals]
yp = np.arange(len(m_labels))
hb = ax.barh(yp, pvals, color=bcolors, alpha=0.85, height=0.58, zorder=3)
ax.axvline(0.05, color='#C0392B', lw=0.9, linestyle='--', zorder=4)
ax.axvline(0.01, color=C_MID,    lw=0.9, linestyle='--', zorder=4)
ax.set_yticks(yp); ax.set_yticklabels(m_labels, fontsize=6.8)
ax.set_xlabel('p-value  (Mann-Whitney U test)')
ax.set_xlim(0, 0.060)
ax.set_title('Fig. 8  —  Statistical Significance', fontweight='bold', pad=5)
for bar, p in zip(hb, pvals):
    sig = '**' if p<0.01 else '*'
    ax.text(p+0.0007, bar.get_y()+bar.get_height()/2,
            f'p={p:.4f} {sig}', va='center', fontsize=6, fontweight='bold')
leg = [Line2D([0],[0], color='#C0392B', lw=0.9, ls='--', label='α = 0.05'),
       Line2D([0],[0], color=C_MID,    lw=0.9, ls='--', label='α = 0.01'),
       mpatches.Patch(fc=C_ZTA, alpha=0.85, label='p < 0.01 (**)'),
       mpatches.Patch(fc=C_MID, alpha=0.85, label='0.01 ≤ p < 0.05 (*)')]
ax.legend(handles=leg, loc='lower right', framealpha=0.92, edgecolor='#CCC',
          fontsize=6, handlelength=1.0, handletextpad=0.4)
ax.xaxis.grid(True, linestyle='--', linewidth=0.35, alpha=0.5)
ax.set_axisbelow(True); ax.invert_yaxis()
plt.tight_layout()
save('fig8_stats.png')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 9 — Detection Quality (single column)
# ══════════════════════════════════════════════════════════════════════════════
dq_l   = ['TPR','Timely\nDetect','TNR','Precision','F1']
per_dq = [100.0,   0.0, 92.0, 92.0, 95.8]
zta_dq = [100.0, 100.0, 96.2, 96.2, 98.1]
x3 = np.arange(len(dq_l)); w3 = 0.32

fig, ax = plt.subplots(figsize=(W1, 2.65))
bp3 = ax.bar(x3-w3/2, per_dq, w3, color=C_PER, alpha=0.85, label='Perimeter')
bz3 = ax.bar(x3+w3/2, zta_dq, w3, color=C_ZTA, alpha=0.85, label='ZTA')

for bar, v in zip(bp3, per_dq):
    y = v+1 if v>0 else 2.5
    ax.text(bar.get_x()+bar.get_width()/2, y,
            f'{v:.0f}%', ha='center', va='bottom',
            fontsize=6, fontweight='bold', color='#7B241C')
for bar, v in zip(bz3, zta_dq):
    ax.text(bar.get_x()+bar.get_width()/2, v+1,
            f'{v:.0f}%', ha='center', va='bottom',
            fontsize=6, fontweight='bold', color='#1E8449')

ax.annotate('Perimeter: 0%\ntimely alerts',
            xy=(x3[1]-w3/2, 4), xytext=(2.15, 42),
            fontsize=6, color='#7B241C', ha='center',
            arrowprops=dict(arrowstyle='->', color='#7B241C', lw=0.8,
                            connectionstyle='arc3,rad=0.25'))

ax.set_xticks(x3); ax.set_xticklabels(dq_l)
ax.set_ylim(0, 118)
ax.set_ylabel('Score (%)')
ax.set_title('Fig. 9  —  Detection Quality Metrics', fontweight='bold', pad=5)
ax.legend(loc='lower right', framealpha=0.92, edgecolor='#CCC',
          handlelength=1.2, handletextpad=0.4)
ax.yaxis.grid(True, linestyle='--', linewidth=0.35, alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
save('fig9_detection_quality.png')

# ─────────────────────────────────────────────────────────────────────────────
print(f'\n✅  All 8 figures saved to: {IMG}')
print('   fig2_mttd.png              MTTD log-scale')
print('   fig3_visibility.png        Visibility + SIEM breakdown')
print('   fig4_mttc_br.png           MTTC log-scale + Breach Radius')
print('   fig5_fpr.png               False Positive Rate')
print('   fig6_overhead.png          Auth Latency + Throughput')
print('   fig7_radar.png             Security profile radar')
print('   fig8_stats.png             Mann-Whitney p-values')
print('   fig9_detection_quality.png Detection quality')

Generating figures...
  ✓ fig2_mttd.png
  ✓ fig3_visibility.png
  ✓ fig4_mttc_br.png
  ✓ fig5_fpr.png
  ✓ fig6_overhead.png
  ✓ fig7_radar.png
  ✓ fig8_stats.png
  ✓ fig9_detection_quality.png

✅  All 8 figures saved to: /content/drive/MyDrive/zta_project/images
   fig2_mttd.png              MTTD log-scale
   fig3_visibility.png        Visibility + SIEM breakdown
   fig4_mttc_br.png           MTTC log-scale + Breach Radius
   fig5_fpr.png               False Positive Rate
   fig6_overhead.png          Auth Latency + Throughput
   fig7_radar.png             Security profile radar
   fig8_stats.png             Mann-Whitney p-values
   fig9_detection_quality.png Detection quality
